# UTS DL – training di Colab (Task 2 & 3, opsional Task 1)
**Cara pakai:** Runtime → Change runtime type → **T4 GPU**, lalu **Runtime → Run all**.

- Semua output (checkpoint, log, grafik) disimpan ke Google Drive (`OUT_DIR`). Jika sesi terputus, cukup **Run all** lagi: (model, fold) yang sudah selesai dilewati dan training yang terputus dilanjutkan dari checkpoint epoch terakhir.
- Hasil final untuk repo ditulis ke `OUT_DIR/repo/` dengan struktur yang sama seperti repo (`prediksi_rul.csv`, `prediksi/`, `results/task2/`, `models/`), lalu di-zip menjadi `OUT_DIR/hasil_repo.zip` untuk diunduh.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# --- Sesuaikan bila perlu ---
REPO_URL = 'https://github.com/roplyhmn/UTS-Deep-Learning.git'
BRANCH   = 'task2/rul'                                  # branch yang sedang dikerjakan
ZIP_PATH = '/content/drive/MyDrive/data_uts_dl.zip'     # zip resmi dari dosen
OUT_DIR  = '/content/drive/MyDrive/uts_dl_outputs'      # hasil di Drive (aman jika sesi putus)
DATA_DIR = '/content/data'
REPO_OUT = f'{OUT_DIR}/repo'                            # cermin struktur repo untuk hasil final
RUN_TASK1 = False                                       # notebook Alief via run_all (lama; opsional)
RUN_TASK2 = True
RUN_W100  = False                                       # eksperimen window 100 (tidak menimpa CSV final)
RUN_TASK3 = False                                       # diaktifkan setelah Task 3 siap
COMMON = f'--data_dir {DATA_DIR} --out_dir {OUT_DIR} --pred_dir {REPO_OUT} --results_dir {REPO_OUT}/results --models_dir {REPO_OUT}/models'

In [ ]:
import os
if not os.path.exists('/content/UTS-Deep-Learning'):
    !git clone -q -b $BRANCH $REPO_URL /content/UTS-Deep-Learning
else:
    !git -C /content/UTS-Deep-Learning fetch -q origin $BRANCH && git -C /content/UTS-Deep-Learning checkout -q $BRANCH && git -C /content/UTS-Deep-Learning reset -q --hard origin/$BRANCH
%cd /content/UTS-Deep-Learning
!git log --oneline -1
!pip install -q -r kode/requirements.txt

In [ ]:
# Ekstrak data (zip bisa berisi folder 'data/' atau berkas langsung - keduanya ditangani)
import zipfile, glob, shutil
if not os.path.exists(f'{DATA_DIR}/rul_train.csv.gz'):
    os.makedirs(DATA_DIR, exist_ok=True)
    zipfile.ZipFile(ZIP_PATH).extractall('/content/_unzip')
    for f in glob.glob('/content/_unzip/**/*.*', recursive=True):
        shutil.move(f, os.path.join(DATA_DIR, os.path.basename(f)))
print(sorted(os.listdir(DATA_DIR)))

In [ ]:
# Cek GPU + catat versi library (untuk laporan)
import torch, json
assert torch.cuda.is_available(), 'GPU tidak aktif: Runtime -> Change runtime type -> T4 GPU'
os.makedirs(REPO_OUT, exist_ok=True)
!python -c "import json; from kode.common.utils import env_info; i=env_info(); print(json.dumps(i, indent=1)); json.dump(i, open('$OUT_DIR/env_info.json','w'), indent=1)"

In [ ]:
# Task 1 (opsional): eksekusi notebook Alief tanpa mengubah isinya
if RUN_TASK1:
    !python -m kode.task1.main {COMMON} --task1_zip "{ZIP_PATH}"

In [ ]:
# Uji Task 2: anti-kebocoran (~10 detik) + model/MC dropout (~5 detik)
if RUN_TASK2:
    !python -m kode.task2.test_leakage --data_dir $DATA_DIR
    !python -m kode.task2.test_models

In [ ]:
# Uji cepat seluruh pipeline Task 2 (subset kecil, < 1 menit; hasil ke OUT_DIR/task2/debug)
if RUN_TASK2:
    !python -m kode.task2.main --debug --no-eda --data_dir $DATA_DIR --out_dir $OUT_DIR

In [ ]:
# Task 2 PENUH: EDA (a-c) -> baseline + LSTM + TCN + Transformer, 5 fold (d-e)
# -> interval MC dropout + kebijakan servis (f). Resume otomatis.
if RUN_TASK2:
    !python -m kode.task2.main {COMMON}

In [ ]:
# Eksperimen: window W=100. Hasil di OUT_DIR/task2/models_W100, CSV final tidak ditimpa
if RUN_TASK2 and RUN_W100:
    !python -m kode.task2.main --W 100 --no-write_pred --no-eda {COMMON}

In [ ]:
# Ringkasan hasil Task 2
import pandas as pd
from IPython.display import Image, display
if RUN_TASK2:
    r = f'{REPO_OUT}/results/task2'
    display(pd.read_csv(f'{r}/models/tabel_perbandingan_model.csv'))
    display(pd.read_csv(f'{r}/models/tabel_per_rentang_rul.csv').pivot(index='rentang_rul', columns='model', values='rmse'))
    display(pd.read_csv(f'{r}/models/tabel_per_mode.csv'))
    display(pd.read_csv(f'{r}/uncertainty/tabel_kalibrasi_interval.csv'))
    display(pd.read_csv(f'{r}/uncertainty/tabel_kebijakan_servis.csv'))
    for g in ['models/10_rmse_per_rentang', 'models/11_prediksi_vs_aktual', 'models/12_kurva_training',
              'uncertainty/13_kalibrasi_interval', 'uncertainty/14_contoh_interval_val', 'uncertainty/15_kebijakan_servis']:
        display(Image(f'{r}/{g}.png', width=800))
    if RUN_W100 and os.path.exists(f'{OUT_DIR}/task2/models_W100/tabel_perbandingan_model.csv'):
        print('W=100:'); display(pd.read_csv(f'{OUT_DIR}/task2/models_W100/tabel_perbandingan_model.csv'))

In [ ]:
# Validasi format CSV prediksi + paket hasil untuk repo
!python -m kode.validasi_csv --data_dir $DATA_DIR --pred_dir $REPO_OUT
!mkdir -p $REPO_OUT/results/task2 && cp $OUT_DIR/env_info.json $REPO_OUT/results/task2/env_info.json   # versi library run ini
shutil.make_archive(f'{OUT_DIR}/hasil_repo', 'zip', REPO_OUT)
print('Unduh:', f'{OUT_DIR}/hasil_repo.zip')

Setelah selesai: unduh `uts_dl_outputs/hasil_repo.zip` dari Drive dan beri tahu Claude lokasinya; isinya disalin ke repo (`prediksi_rul.csv`, `prediksi/`, `results/`, `models/`).